# Initial QC

+ Plate 1: 96 samples(60/36), on MEGA kit v2, seq 15 sub-lib X 3 runs
+ Plate 2: 96 samples (88/8), on MEGA kit v2, seq 16 sub-lib X 3 runs
+ ~30 Billion reads per plate
+ Parse workflow has a low [doublet rate](https://support.parsebiosciences.com/hc/en-us/articles/360053107311-What-is-the-expected-doublet-rate#:~:text=Doublet%20rates%20are%20low%2C%20less,through%20the%20Whole%20Transcriptome%20workflow.): ~3% per 100K cells

***

- Moved to Scanpy from Seurat as combined object size of both plates was too much for R
- FC samples included
- 3 filtering steps
- Rm cells with < 500 genes or > 6000 genes
- Rm doublets (by sublibrary)
- Rm cells > 5% reads map to mito or ribo genes
- Rm genes exp in < 5 cells
- Rm genes [MitoCarta 3.0](https://academic.oup.com/nar/article/49/D1/D1541/5974091) genes
  - Downloaded [Human.MitoCarta3.0.xls](https://personal.broadinstitute.org/scalvo/MitoCarta3.0/Human.MitoCarta3.0.xls) from the [Broad Institute site](https://www.broadinstitute.org/files/shared/metabolism/mitocarta/human.mitocarta3.0.html) 
- Rm samples with < 10 cells (probs need to up this)


In [ ]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

In [4]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables and filters
resolutions = [0.1, 0.3]
batch_col = 'plate' # Should we set to plate and sample??
rm_doublets = True
rm_mad_outliers = False
genes_per_cell_thresh = {'min': 750, 'max': 5000}
counts_per_cell_thresh = {'min': 500, 'max': None}
mito_ribo_thresh = {'mito': 5, 'ribo': 5}
cells_per_gene_thresh = 5
cells_per_smpl_thresh = 10

# Load data

In [ ]:
# Initialize the environment and get all paths and logger
%pip install xlrd # Run once on Hawk
logger, root_dir, sheets_dir, plate_path, scanpy_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = load_and_dwnsmpl_data(None, plate_path)
mito_genes = pd.read_excel(sheets_dir + 'Human.MitoCarta3.0.xls', sheet_name = 'A Human MitoCarta3.0')['Symbol'].tolist()
adata.obs['sample'] = adata.obs['sample'].str.replace('sample_', '')
adata = adata[~adata.obs['sample'].str.endswith(tuple(['WGE', 'Hipp', 'Thal']))]
adata.obs['sublibrary'] = [x[1] for x in adata.obs.index.str.split('__s')] 

# Initial cell counts

In [ ]:
# Cell counts by sample
print(f"Number of samples: {adata.obs['sample'].nunique()}")
adata.obs['sample'].value_counts()

In [ ]:
# Cell counts by sublibrary
adata.obs['sublibrary'].value_counts()

In [ ]:
adata.obs

# QC metadata

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)
adata.obs

In [ ]:
adata.obs[['tscp_count', 'total_counts']] # Why the discrepancy? Duplicates? This is the filtered matrix??!!

In [ ]:
sample_qc_metrics = (
    adata.obs.groupby("sample")
    .agg(
        median_genes_per_cell=("n_genes_by_counts", "median"),
        median_counts_per_cell=("total_counts", "median"),
        total_cells=("n_genes_by_counts", "count"),
    )
    .reset_index()
)
sample_qc_metrics

In [ ]:
# Most experessed genes - Not such a big deal now?
logger.info("Most exp genes ...")
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
# Filter 1

#### NOTE: tscp_count != total_counts. Scanpy uses total_counts for filters. Check why these values don't match ####
logger.info("Applying filter 1 ...")
filter_anndata(
    adata, 
    min_genes_per_cell=genes_per_cell_thresh['min'], 
    max_genes_per_cell=genes_per_cell_thresh['max'],
    min_counts_per_cell=counts_per_cell_thresh['min'], 
    min_cells_per_gene=cells_per_gene_thresh, 
    min_cells_per_sample=cells_per_smpl_thresh)
adata.obs['sample'].value_counts() 


In [ ]:
# Filter 2
# - Doublets
if rm_doublets is True:
    logger.info("Detecting Doublets ...")
    sc.external.pp.scrublet(adata, batch_key = 'sublibrary') # This fails on Hawk 

In [ ]:
# Code for Scrublet: failed on Hawk due to memory issues
if rm_doublets is True:
    logger.info("Plotting doublets ...")   
    plot_doublet_umaps(adata)

In [13]:
if rm_mad_outliers is True:
    logger.info("Detecting outliers ...")

    detect_mad_outliers_per_sample(
        adata,
        group_column="sample",       # Column in `adata.obs` to group by
        target_column="total_counts",  # Column to detect outliers
        threshold=3,                # Number of MADs for outlier detection
        log=False,                  # Whether to log-transform the data
        use_median=True             # Use median and MAD (or mean and SD)
    )
    
    # Create the violin plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sc.pl.violin(
        adata,
        keys='total_counts',
        jitter=0.4,
        groupby='sample',
        rotation=90,
        size=1.5,
        ax=ax,
        show=False, 
        color='Red'
    )
    
    # Overlay outlier cells with red dots
    outliers = adata.obs.loc[adata.obs['mad_outlier']]
    ax.scatter(
        x=outliers['sample'],  # X-coordinates (groupby value)
        y=outliers['total_counts'],  # Y-coordinates (outlier values)
        color='red',
        label='Outliers',
        s=10,
        alpha=0.8
    )


In [ ]:
logger.info("Applying filter 2 ...")

filter_cells_and_genes(
    adata,
    mito_threshold=5,
    ribo_threshold=5,
    remove_doublets=True,
    remove_mad_outliers=False,
    genes_to_remove=[mito_genes, 'MALAT1']  # New parameter
)

In [ ]:
# Plot UMAP
def run_default_scanpy(ann_obj):

    sc.pp.normalize_total(ann_obj) # Norm to median total count
    sc.pp.log1p(ann_obj)
    sc.pp.highly_variable_genes(ann_obj, n_top_genes=2000, flavor="seurat_v3")
    sc.tl.pca(ann_obj, svd_solver='arpack')
    sc.pp.neighbors(ann_obj)
    sc.tl.leiden(ann_obj)
    sc.pl.umap(ann_obj, color=['leiden'])

run_default_scanpy(adata)

In [ ]:
# Plot vlns
gene_sets = [
    ("general_genes", general_genes),
    ("pfc_features", pfc_features)
]

fig = plot_filtered_violin(
    adata, 
    gene_sets, 
    groupby_base="leiden", 
    resolutions=None, 
    clustering_algorithm="Leiden")
plt.show()  # Display the figure

In [ ]:
# Plot sample per cluster
# Function saves an excel file with the cell counts per sample per cluster 
# Extract sample and leiden cluster information from the AnnData object
fig = plot_and_save_cluster_percentages(
    adata=adata,
    output_dir = scanpy_dir,
    clustering_param="leiden"
)
plt.show() 

In [ ]:
# Final Dimesnsions
adata.shape

In [ ]:
# Cells per sample after filter
adata.obs['sample'].value_counts()

In [ ]:
# Final distributions per sample
# Create the violin plot
fig, ax = plt.subplots(figsize=(10, 6))
sc.pl.violin(
    adata,
    keys='gene_count',
    jitter=0.4,
    groupby='sample',
    rotation=90,
    size=1.5,
    ax=ax,
    show=False, 
    color='Red'
)
plt.title('Final gene per cell')
plt.axhline(y = 1000, color = 'r', linestyle = '-')
plt.axhline(y = 6000, color = 'r', linestyle = '-') 

fig, ax = plt.subplots(figsize=(10, 6))
sc.pl.violin(
    adata,
    keys='total_counts',
    jitter=0.4,
    groupby='sample',
    rotation=90,
    size=1.5,
    ax=ax,
    show=False, 
    color='Red'
)
plt.title('Final counts per cell')
plt.axhline(y = 500, color = 'r', linestyle = '-')

In [ ]:
# Save
logger.info("Saving h5ad file ...")
adata.write(scanpy_dir + f'adata_qc_{plate}.h5ad')
logger.info(f"{plate} qc run done.")